In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pdfplumber faiss-cpu transformers sentencepiece -q

import os, json
import pdfplumber
from tqdm import tqdm

In [ ]:
# ==========================
# 1) PDF → TEXT 추출
# ==========================

import os
import pdfplumber

def extract_text_from_pdf(pdf_path: str) -> str:
    """PDF 전체 텍스트 추출"""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"{pdf_path} not found: 현재 경로 = {os.getcwd()}")

    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text() or ""
            txt = txt.replace("\u00A0", " ")
            lines.append(txt)
        print(f"총 {len(pdf.pages)} 페이지에서 텍스트 추출 완료")

    return "\n".join(lines)

In [ ]:
base = "/content/drive/MyDrive/rag-mmlu-ewha/data/external_kb"

domains = {
    "law": [
        f"{base}/외부문서(Law)_introlaw.pdf",
        f"{base}/외부문서(Law)_OpenCourseWare.pdf"
    ],

    "psychology": [
        f"{base}/외부문서(History)_openstaxworldhistory.pdf"
    ],

    "business": [
        f"{base}/외부문서(Business)_Marketing.pdf",
        f"{base}/외부문서(Business)_OpenStaxEconomics.pdf"
    ],

    "philosophy": [
        f"{base}/외부문서(Philosophy)_철학개론.pdf"
    ],

    "history": [
        f"{base}/외부문서(History)_openstaxworldhistory.pdf"
    ]
}

# 실제 PDF에서 텍스트 뽑아서 변수에 담기
documents = []   # [(domain, text), ...]

for domain, filelist in domains.items():
    for pdf_path in filelist:
        print(f"Processing [{domain}] {pdf_path}")
        text = extract_text_from_pdf(pdf_path)
        documents.append((domain, text))

Processing [law] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Law)_introlaw.pdf
총 32 페이지에서 텍스트 추출 완료
Processing [law] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Law)_OpenCourseWare.pdf
총 27 페이지에서 텍스트 추출 완료
Processing [psychology] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(History)_openstaxworldhistory.pdf
총 64 페이지에서 텍스트 추출 완료
Processing [business] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Business)_Marketing.pdf
총 10 페이지에서 텍스트 추출 완료
Processing [business] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Business)_OpenStaxEconomics.pdf
총 58 페이지에서 텍스트 추출 완료
Processing [philosophy] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(Philosophy)_철학개론.pdf
총 8 페이지에서 텍스트 추출 완료
Processing [history] /content/drive/MyDrive/rag-mmlu-ewha/data/external_kb/외부문서(History)_openstaxworldhistory.pdf
총 64 페이지에서 텍스트 추출 완료


In [ ]:
from collections import Counter

domain_counts = Counter([d for d, _ in documents])
print("도메인별 문서 개수:", domain_counts)

print("=== documents 샘플 2개 출력 ===")
for i in range(min(2, len(documents))):
    domain, text = documents[i]
    print(f"[{i}] domain = {domain}")
    print(text[:500], "...")  # 앞 500자만 미리보기
    print("=" * 80)

print("=== 각 문서 word 길이 확인 ===")
for i, (domain, text) in enumerate(documents):
    word_count = len(text.split())
    print(f"[{i}] domain={domain} | words={word_count}")

도메인별 문서 개수: Counter({'law': 2, 'business': 2, 'psychology': 1, 'philosophy': 1, 'history': 1})
=== documents 샘플 2개 출력 ===
[0] domain = law

Established in 1931, the Institute of Government provides training, advisory, and research
services to public officials and others interested in the operation of state and local
government in North Carolina. A part of The University of North Carolina at Chapel Hill,
the Institute also administers the university’s Master of Public Administration Program.
Each year approximately 14,000 city, county, and state officials attend one or more of the
230 classes, seminars, and conferences offered by th ...
[1] domain = law







Karl Marx (1818-1883). From the Preface to A Contribution to the Critique of Political
Economy. (1859) 1913, pp. 11-13.
Herbert, Bob. “Nike's Boot Camps.” In America. In New York Times. 31 Mar. 1997.
Gayle Krishenbanm. “Nike's Nemesis." In Newsmaker: Cicih Sukaesih.
Griswold, Wendy. "The Ideas of the Reading Class." Contemporary S

In [ ]:
# TEXT → CHUNKS 변환

# 300 tokens(=약 200~300자) 전후가 RAG에서 가장 많이 쓰는 sweet spot
# 250~350 token chunk
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

kb = []
doc_id = 1

for domain, text in documents:
    chunks = chunk_text(text)
    for c in chunks:
        kb.append({
            "doc_id": f"{domain}-{doc_id}",
            "domain": domain,
            "source": "external_kb",
            "text": c
        })
        doc_id += 1

output_jsonl = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl"

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in kb:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_jsonl)
print("총 chunks:", len(kb))
'''
외부문서는 LangChain LLM cleanup 안한 이유
1. 민감영역 -> 교수님이 외부문서는 직접 찾아서 쓰라고 했으므로
llm으로 정제하기 민감
2. 성능 측면에서도 chunk만 하는 걸 추천
RAG에서는 "잘 정제된 문장"보다 "많은 coverage"가 더 중요
'''

Saved: /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl
총 chunks: 237


'\n외부문서는 LangChain LLM cleanup 안한 이유\n1. 민감영역 -> 교수님이 외부문서는 직접 찾아서 쓰라고 했으므로\nllm으로 정제하기 민감\n2. 성능 측면에서도 chunk만 하는 걸 추천\nRAG에서는 "잘 정제된 문장"보다 "많은 coverage"가 더 중요\n'

In [ ]:
output_jsonl = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl"

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in kb:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_jsonl)
print("총 chunks:", len(kb))

Saved: /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb.jsonl
총 chunks: 237


In [ ]:
# 임베딩 모델
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

class EmbeddingModel:
    def __init__(self, model_name="BAAI/bge-m3"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)

    @torch.no_grad()
    def encode(self, texts):
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(self.device)

        outputs = self.model(**inputs)
        dense = outputs.last_hidden_state[:, 0]
        dense = dense / dense.norm(dim=1, keepdim=True)
        return dense.cpu().numpy()


In [ ]:
# FAISS Index 생성

import faiss

texts = [row["text"] for row in kb]
model = EmbeddingModel()

print("Encoding MMLU KB...")
vectors = model.encode(texts)

index_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb_index.faiss"

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
faiss.write_index(index, index_path)

print("Saved index →", index_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Encoding MMLU KB...
Saved index → /content/drive/MyDrive/rag-mmlu-ewha/data/mmlu_kb_index.faiss
